# LeetCode #211 — Design Add and Search Words Data Structure: trie + DFS cho ký tự đại diện

**Mức độ:** Medium · **Chủ đề:** String, Depth-First Search, Design, Trie

Skeleton trên LeetCode:

```python
class WordDictionary:
    def __init__(self):
    def addWord(self, word: str) -> None:
    def search(self, word: str) -> bool:
```

Mục tiêu của notebook:

1. Dùng lại **trie** (#208) để lưu từ.
2. Xử lý ký tự đại diện `'.'` (khớp **một** ký tự bất kỳ): tại nút đó phải **rẽ nhánh** sang mọi con → **DFS / quay lui** trên trie.
3. Phân tích độ phức tạp: chữ thường đi một đường, dấu chấm có thể rẽ tối đa 26 nhánh.
4. So sánh với các cách khác: duyệt mọi từ, nhóm theo độ dài, regex.

# 1. Bài toán là gì?

Cài đặt `WordDictionary`:

- `addWord(word)`: thêm `word`.
- `search(word)`: `True` nếu có một từ đã thêm **khớp** `word`, trong đó `'.'` khớp với **đúng một** ký tự bất kỳ.

**Constraints (theo đề):**

- `1 <= word.length <= 25`
- `word` trong `addWord` gồm chữ thường; `word` trong `search` gồm chữ thường hoặc `'.'`.
- Mỗi truy vấn `search` có **tối đa 2** dấu `'.'`.
- Tối đa `10^4` lời gọi.

---

## Ví dụ

```text
addWord("bad"); addWord("dad"); addWord("mad")
search("pad")  -> False
search("bad")  -> True
search(".ad")  -> True
search("b..")  -> True
search("b.")   -> False      (độ dài phải khớp chính xác)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Trie (cây tiền tố)
2. Quay lui (backtracking)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Trie (cây tiền tố)

### Trie là gì?

Trie là cây trong đó mỗi **cạnh** mang **một ký tự**; đường đi từ gốc tới một nút tạo thành một **tiền tố**. Các từ có chung tiền tố **dùng chung** đường đi. Mỗi nút có cờ **kết thúc** cho biết tiền tố đó có phải một từ đã thêm không.

```text
thêm: "car", "cat", "cart", "dog"

          (gốc)
          /   \
         c     d
         |     |
         a     o
        / \    |
       r*  t*  g*
       |
       t*            * = kết thúc một từ
```

### Ba thao tác cơ bản — đều O(độ dài chuỗi)

In [1]:
class TrieNodeDemo:
    def __init__(self):
        self.children = {}          # ký tự → nút con
        self.end = False


def trie_insert(root, word):
    node = root
    for ch in word:
        if ch not in node.children:
            node.children[ch] = TrieNodeDemo()
        node = node.children[ch]
    node.end = True


def trie_walk(root, s):
    node = root
    for ch in s:
        node = node.children.get(ch)
        if node is None:
            return None
    return node


root_demo = TrieNodeDemo()
for w in ["car", "cat", "cart", "dog"]:
    trie_insert(root_demo, w)

for q in ["car", "ca", "cart", "cab"]:
    nd = trie_walk(root_demo, q)
    print(f"{q!r:7}: là tiền tố? {str(nd is not None):<5} | là từ? {nd is not None and nd.end}")

'car'  : là tiền tố? True  | là từ? True
'ca'   : là tiền tố? True  | là từ? False
'cart' : là tiền tố? True  | là từ? True
'cab'  : là tiền tố? False | là từ? False


### Liệt kê mọi từ có một tiền tố (gợi ý tự động)

Đi tới nút của tiền tố, rồi **DFS** toàn bộ cây con.

In [2]:
def words_with_prefix(root, prefix):
    start = trie_walk(root, prefix)
    out = []
    def dfs(node, path):
        if node.end:
            out.append(path)
        for ch in sorted(node.children):
            dfs(node.children[ch], path + ch)
    if start:
        dfs(start, prefix)
    return out

print(words_with_prefix(root_demo, "ca"), words_with_prefix(root_demo, "x"))

['car', 'cart', 'cat'] []


### Cài đặt gọn bằng dict lồng nhau

Mỗi nút là một `dict`; dùng một khóa đặc biệt (ví dụ `"$"`, không phải chữ cái) làm cờ kết thúc.

In [3]:
root_d = {}
for w in ["car", "cat"]:
    node = root_d
    for ch in w:
        node = node.setdefault(ch, {})
    node["$"] = True
print(root_d)

{'c': {'a': {'r': {'$': True}, 't': {'$': True}}}}


### So sánh với `set` các từ

| Thao tác | `set` | Trie |
|---|---|---|
| có từ `w`? | O(L) trung bình | O(L) |
| có từ nào bắt đầu bằng `p`? | O(N · L) — duyệt mọi từ | **O(len(p))** |
| liệt kê từ theo tiền tố, thứ tự từ điển | sắp xếp lại | DFS theo thứ tự con |

### Lỗi hay gặp

- Không có cờ kết thúc → không phân biệt "từ" với "tiền tố".
- Tạo nút mới **đè** nút con đã có (`children[ch] = Node()` vô điều kiện) → mất nhánh cũ.
- Cờ kết thúc dùng khóa trùng với một ký tự hợp lệ.

## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [4]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [5]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [6]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [7]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [8]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Áp dụng vào bài này

Lưu từ trong **trie**. Ký tự thường đi đúng một nhánh, còn dấu `'.'` phải **thử mọi nhánh con** — tìm kiếm trở thành **quay lui** trên trie: thử một nhánh, thất bại thì quay về thử nhánh kế tiếp; tới cuối mẫu thì cần cờ kết thúc.

# 3. Python cơ bản cần biết

In [9]:
from typing import List
import random, time, re

def matches(word, pattern):
    """Khớp ký tự-đối-ký tự: '.' khớp mọi ký tự, độ dài phải bằng nhau."""
    return len(word) == len(pattern) and all(p == "." or p == c for c, p in zip(word, pattern))

print(matches("bad", ".ad"), matches("bad", "b."), matches("bad", "b.d"))
print("any() dừng ngay khi gặp True:", any(x > 2 for x in [1, 3, 10**9]))

True False True
any() dừng ngay khi gặp True: True


# 4. Baseline: lưu danh sách, so khớp từng từ

In [10]:
class WordDictionaryList:
    def __init__(self):
        self.words = []
    def addWord(self, word):
        self.words.append(word)
    def search(self, word):
        return any(matches(w, word) for w in self.words)       # O(N · L) mỗi truy vấn

wl = WordDictionaryList()
for w in ["bad", "dad", "mad"]:
    wl.addWord(w)
print([wl.search(q) for q in ["pad", "bad", ".ad", "b..", "b."]])

[False, True, True, True, False]

Mỗi `search` duyệt **mọi** từ. Một cải tiến dễ: nhóm từ theo độ dài (chỉ so với từ cùng độ dài) — nhưng nếu các từ cùng độ dài thì vẫn chậm. Trie cho phép **cắt tỉa**: chữ cái cụ thể chỉ đi đúng một nhánh.

# 5. Ý tưởng chính: trie + DFS

```text
dfs(node, i):                   # đã khớp word[:i], đang ở node
    if i == len(word): return node có cờ kết thúc
    c = word[i]
    if c == '.':
        return any(dfs(child, i + 1) for child in node.children)    # thử mọi nhánh
    return c in node.children and dfs(node.children[c], i + 1)     # chỉ một nhánh
```

Đây là **quay lui** trên trie: ở dấu chấm ta thử từng con, nhánh nào thất bại thì quay về thử nhánh tiếp theo.

In [11]:
class WordDictionaryTrie:
    def __init__(self):
        self.root = {}

    def addWord(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word: str) -> bool:
        def dfs(node, i):
            if i == len(word):
                return "$" in node
            c = word[i]
            if c == ".":
                return any(dfs(child, i + 1) for key, child in node.items() if key != "$")
            return c in node and dfs(node[c], i + 1)
        return dfs(self.root, 0)

wt = WordDictionaryTrie()
for w in ["bad", "dad", "mad"]:
    wt.addWord(w)
print([wt.search(q) for q in ["pad", "bad", ".ad", "b..", "b."]])

[False, True, True, True, False]


# 6. Mô phỏng

In [12]:
def trace_search(wd, word):
    calls = [0]
    def dfs(node, i, path):
        calls[0] += 1
        pad = "   " * i
        if i == len(word):
            ok = "$" in node
            print(pad + f"hết mẫu tại '{path}': {'là từ ✔' if ok else 'không phải từ ✘'}")
            return ok
        c = word[i]
        if c == ".":
            kids = sorted(k for k in node if k != "$")
            print(pad + f"'.' tại '{path}' → thử các nhánh {kids}")
            for k in kids:
                if dfs(node[k], i + 1, path + k):
                    return True
            return False
        if c not in node:
            print(pad + f"'{path}' không có con '{c}' ✘")
            return False
        return dfs(node[c], i + 1, path + c)
    r = dfs(wd.root, 0, "")
    print(f"search('{word}') = {r}, số lần gọi dfs = {calls[0]}\n")

wt2 = WordDictionaryTrie()
for w in ["bad", "bat", "dad", "mad", "mud"]:
    wt2.addWord(w)
trace_search(wt2, ".ud")
trace_search(wt2, "..x")

'.' tại '' → thử các nhánh ['b', 'd', 'm']
   'b' không có con 'u' ✘
   'd' không có con 'u' ✘
         hết mẫu tại 'mud': là từ ✔
search('.ud') = True, số lần gọi dfs = 6

'.' tại '' → thử các nhánh ['b', 'd', 'm']
   '.' tại 'b' → thử các nhánh ['a']
      'ba' không có con 'x' ✘
   '.' tại 'd' → thử các nhánh ['a']
      'da' không có con 'x' ✘
   '.' tại 'm' → thử các nhánh ['a', 'u']
      'ma' không có con 'x' ✘
      'mu' không có con 'x' ✘
search('..x') = False, số lần gọi dfs = 8



`"..x"` phải đi hết mọi nhánh độ sâu 2 rồi mới kết luận `False` — đó là trường hợp xấu của dấu chấm.

# 7. Phiên bản đúng format LeetCode

Dùng DFS **lặp** với ngăn xếp `(node, i)` — tránh tạo generator lồng nhau và dễ đọc.

In [13]:
class WordDictionary:
    def __init__(self):
        self.root = {}

    def addWord(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word: str) -> bool:
        n = len(word)
        stack = [(self.root, 0)]
        while stack:
            node, i = stack.pop()
            if i == n:
                if "$" in node:
                    return True
                continue
            c = word[i]
            if c == ".":
                for key, child in node.items():
                    if key != "$":
                        stack.append((child, i + 1))
            elif c in node:
                stack.append((node[c], i + 1))
        return False

In [14]:
def run_ops(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if op == "WordDictionary":
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


OPS = ["WordDictionary", "addWord", "addWord", "addWord", "search", "search", "search", "search"]
ARGS = [[], ["bad"], ["dad"], ["mad"], ["pad"], ["bad"], [".ad"], ["b.."]]
EXP = [None, None, None, None, False, True, True, True]
check("Ví dụ", run_ops(WordDictionary, OPS, ARGS), EXP)
check("Ví dụ (đệ quy)", run_ops(WordDictionaryTrie, OPS, ARGS), EXP)

random.seed(211)
for _ in range(400):
    ops, args = ["WordDictionary"], [[]]
    for _ in range(random.randint(1, 25)):
        if random.random() < 0.5:
            ops.append("addWord"); args.append(["".join(random.choice("ab") for _ in range(random.randint(1, 3)))])
        else:
            ops.append("search"); args.append(["".join(random.choice("ab.") for _ in range(random.randint(1, 3)))])
    exp = run_ops(WordDictionaryList, ops, args)
    assert run_ops(WordDictionary, ops, args) == exp == run_ops(WordDictionaryTrie, ops, args)
print("400 chuỗi thao tác ngẫu nhiên: hai bản trie khớp bản danh sách")

words = ["".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(random.randint(3, 10))) for _ in range(5000)]
queries = []
for _ in range(5000):
    w = list(random.choice(words))
    for p in random.sample(range(len(w)), 2):
        w[p] = "."
    queries.append("".join(w))
for name, cls in [("Trie", WordDictionary), ("Danh sách", WordDictionaryList)]:
    obj = cls()
    for w in words:
        obj.addWord(w)
    t0 = time.time(); found = sum(obj.search(q) for q in queries); t1 = time.time()
    print(f"{name:<10}: 5000 truy vấn (2 dấu chấm) trên 5000 từ → {found} khớp, {t1 - t0:.3f}s")

Ví dụ            PASS  [None, None, None, None, False, True, True, True]


Ví dụ (đệ quy)   PASS  [None, None, None, None, False, True, True, True]
400 chuỗi thao tác ngẫu nhiên: hai bản trie khớp bản danh sách


Trie      : 5000 truy vấn (2 dấu chấm) trên 5000 từ → 5000 khớp, 0.233s


Danh sách : 5000 truy vấn (2 dấu chấm) trên 5000 từ → 5000 khớp, 2.999s


# 8. Vì sao đúng? Độ phức tạp?

**Đúng:** trạng thái `(node, i)` nghĩa là "có từ đã thêm mà tiền tố của nó khớp `word[:i]`, và tiền tố đó là đường đi tới `node`". Chữ cái chỉ có một cách đi tiếp, dấu chấm thử mọi con — vậy DFS liệt kê **đúng** mọi đường đi trie khớp mẫu. Kết quả `True` ⟺ có đường đi khớp đủ `len(word)` ký tự **và** kết thúc ở nút có cờ.

Với `L` = độ dài truy vấn, `d` = số dấu chấm (≤ 2):

```text
addWord:  O(L)
search:   O(26^d · L) trong trường hợp xấu (mỗi dấu chấm nhân tối đa 26 nhánh) — với d ≤ 2 là O(676 · L)
          không có dấu chấm: O(L), như trie thường
Bộ nhớ:   O(tổng độ dài các từ)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Dấu chấm chỉ thử **một** nhánh (nhánh đầu tiên)

In [15]:
class WrongFirstBranch(WordDictionaryTrie):
    def search(self, word):
        node = self.root
        for c in word:
            if c == ".":
                kids = [k for k in node if k != "$"]
                if not kids:
                    return False
                node = node[kids[0]]                # SAI: bỏ qua các nhánh khác
            elif c in node:
                node = node[c]
            else:
                return False
        return "$" in node

ops2 = ["WordDictionary", "addWord", "addWord", "search"]
args2 = [[], ["bad"], ["mud"], [".ud"]]
print("Sai :", run_ops(WrongFirstBranch, ops2, args2))
print("Đúng:", run_ops(WordDictionary, ops2, args2))

Sai : [None, None, None, False]
Đúng: [None, None, None, True]


## Lỗi 2: Quên kiểm tra cờ kết thúc ở cuối mẫu

In [16]:
class WrongNoEnd(WordDictionaryTrie):
    def search(self, word):
        def dfs(node, i):
            if i == len(word):
                return True                          # SAI
            c = word[i]
            if c == ".":
                return any(dfs(ch, i + 1) for k, ch in node.items() if k != "$")
            return c in node and dfs(node[c], i + 1)
        return dfs(self.root, 0)

print("Thêm 'bad', search('b.') — sai:", run_ops(WrongNoEnd, ["WordDictionary", "addWord", "search"], [[], ["bad"], ["b."]])[-1], "| đúng: False")

Thêm 'bad', search('b.') — sai: True | đúng: False


## Lỗi 3: Dấu chấm duyệt cả khóa cờ `"$"` như một nhánh

Nếu dùng dict lồng với khóa `"$"` mà quên loại trừ, `node["$"]` là `True` (không phải dict) → lỗi khi duyệt tiếp.

In [17]:
class WrongIncludeFlag(WordDictionaryTrie):
    def search(self, word):
        def dfs(node, i):
            if i == len(word):
                return "$" in node
            c = word[i]
            if c == ".":
                return any(dfs(ch, i + 1) for ch in node.values())     # SAI: gồm cả True
            return c in node and dfs(node[c], i + 1)
        return dfs(self.root, 0)

try:
    print(run_ops(WrongIncludeFlag, ["WordDictionary", "addWord", "addWord", "search"], [[], ["a"], ["ab"], ["a.."]]))
except (TypeError, AttributeError) as e:
    print(type(e).__name__ + ":", e)

AttributeError: 'bool' object has no attribute 'values'


## Lỗi 4: Dùng regex `re.fullmatch` cho mỗi từ — đúng nhưng vẫn là duyệt mọi từ

`re.fullmatch(".ad", w)` cho kết quả đúng (dấu `.` của regex khớp một ký tự bất kỳ trừ xuống dòng), nhưng nó không giảm số từ phải xét — vẫn `O(N · L)` mỗi truy vấn như baseline.

In [18]:
print([bool(re.fullmatch(".ad", w)) for w in ["bad", "dad", "ba"]])

[True, True, False]


# 10. Mẹo nhận diện pattern

- **Trie + ký tự đại diện / tìm kiếm mờ** → DFS (quay lui) trên trie, rẽ nhánh ở ký tự đại diện.
- **Tìm nhiều từ cùng lúc trong lưới** → trie + DFS trên lưới (#212).
- **Ký tự đại diện khớp chuỗi rỗng / nhiều ký tự** (`*`) → bài so khớp mẫu bằng DP (#44, #10).

Bài liên quan: **#208 Implement Trie**, **#212 Word Search II** (Hard), **#676 Implement Magic Dictionary**, **#745 Prefix and Suffix Search** (Hard), **#44 Wildcard Matching** (Hard), **#1032 Stream of Characters** (Hard).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Sau khi thêm `"cat"`, `"car"`, `"cart"`, `"dog"`: trả lời `search` cho `"ca."`, `"c..t"`, `"...."`, `".o."`, `"d.."`. Đếm số lần gọi DFS cho `"...."`.

## Bài 2 — Điền chỗ trống

In [19]:
class Exercise2WordDictionary:
    def __init__(self):
        self.root = {}

    def addWord(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word: str) -> bool:
        def dfs(node, i):
            # TODO
            pass
        return dfs(self.root, 0)


# assert run_ops(Exercise2WordDictionary, ["WordDictionary", "addWord", "addWord", "addWord", "search", "search", "search", "search"], [[], ["bad"], ["dad"], ["mad"], ["pad"], ["bad"], [".ad"], ["b.."]]) == [None, None, None, None, False, True, True, True]

## Bài 3 — Biến thể: đếm số từ khớp mẫu

In [20]:
class Exercise3CountDictionary:
    def __init__(self):
        # TODO
        pass

    def addWord(self, word: str) -> None:
        pass

    def count(self, pattern: str) -> int:
        pass


# e3 = Exercise3CountDictionary(); [e3.addWord(w) for w in ["bad", "dad", "mad", "ba", "bad"]]; assert e3.count(".ad") == 3 and e3.count("b.") == 1 and e3.count("...") == 3 and e3.count("x..") == 0

Ghi chú: thêm trùng từ `"bad"` hai lần vẫn chỉ tính **một** từ (tập hợp từ).

## Bài 4 — Biến thể: Implement Magic Dictionary (LeetCode #676)

`buildDict(words)`; `search(s)`: `True` nếu đổi **đúng một** ký tự của `s` (sang ký tự khác) thì được một từ trong từ điển.

In [21]:
class Exercise4MagicDictionary:
    def __init__(self):
        # TODO
        pass

    def buildDict(self, dictionary: List[str]) -> None:
        pass

    def search(self, searchWord: str) -> bool:
        pass


# m4 = Exercise4MagicDictionary(); m4.buildDict(["hello", "leetcode"]); assert [m4.search(w) for w in ["hello", "hhllo", "hell", "leetcoded"]] == [False, True, False, False]
# m4b = Exercise4MagicDictionary(); m4b.buildDict(["hello", "hallo"]); assert m4b.search("hello") is True

## Bài 5 — Mini challenge: Word Search II (LeetCode #212, Hard)

Cho lưới chữ cái và danh sách từ, trả về các từ **có thể ghép** bằng đường đi qua các ô kề cạnh (mỗi ô dùng tối đa một lần trong một từ). Gợi ý: dựng trie từ danh sách từ, DFS từ mỗi ô, đi đồng thời trên lưới và trie.

In [22]:
def exercise_5_find_words(board: List[List[str]], words: List[str]) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_5_find_words([["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]], ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
# assert exercise_5_find_words([["a", "b"], ["c", "d"]], ["abcb"]) == []

# 12. Gợi ý

**Bài 1**
- Mức 1: dùng `trace_search` sau khi tự làm.

**Bài 3**
- Mức 1: giống `search` nhưng thay `any` bằng `sum`; nút kết thúc đóng góp 1.

**Bài 4**
- Mức 1: DFS trên trie với thêm tham số `changed` (đã đổi ký tự chưa).
- Mức 2: ở mỗi nút, đi nhánh trùng ký tự với `changed` giữ nguyên; nếu `changed` còn False thì thử mọi nhánh **khác** ký tự với `changed=True`. Cuối cùng cần `changed` là True và nút có cờ.

**Bài 5**
- Mức 1: DFS `(r, c, node)`: ký tự ô phải là con của `node`; đánh dấu ô `'#'` khi đang dùng, khôi phục khi quay lui.
- Mức 2: khi tới nút có từ, thêm vào kết quả và **xóa** cờ để không thêm trùng.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
wd1 = WordDictionaryTrie()
for w in ["cat", "car", "cart", "dog"]:
    wd1.addWord(w)
for q in ["ca.", "c..t", "....", ".o.", "d.."]:
    print(f"search('{q}') = {wd1.search(q)}")
trace_search(wd1, "....")

search('ca.') = True
search('c..t') = True
search('....') = True
search('.o.') = True
search('d..') = True
'.' tại '' → thử các nhánh ['c', 'd']
   '.' tại 'c' → thử các nhánh ['a']
      '.' tại 'ca' → thử các nhánh ['r', 't']
         '.' tại 'car' → thử các nhánh ['t']
            hết mẫu tại 'cart': là từ ✔
search('....') = True, số lần gọi dfs = 5



## Đáp án Bài 2

In [24]:
class Answer2WordDictionary(WordDictionaryTrie):
    pass

assert run_ops(Answer2WordDictionary, ["WordDictionary", "addWord", "addWord", "addWord", "search", "search", "search", "search"], [[], ["bad"], ["dad"], ["mad"], ["pad"], ["bad"], [".ad"], ["b.."]]) == [None, None, None, None, False, True, True, True]
print("Bài 2 PASS (giống WordDictionaryTrie ở mục 5)")

Bài 2 PASS (giống WordDictionaryTrie ở mục 5)


## Đáp án Bài 3

In [25]:
class Answer3CountDictionary:
    def __init__(self):
        self.root = {}

    def addWord(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def count(self, pattern: str) -> int:
        def dfs(node, i):
            if i == len(pattern):
                return 1 if "$" in node else 0
            c = pattern[i]
            if c == ".":
                return sum(dfs(ch, i + 1) for k, ch in node.items() if k != "$")
            return dfs(node[c], i + 1) if c in node else 0
        return dfs(self.root, 0)

e3 = Answer3CountDictionary(); [e3.addWord(w) for w in ["bad", "dad", "mad", "ba", "bad"]]; assert e3.count(".ad") == 3 and e3.count("b.") == 1 and e3.count("...") == 3 and e3.count("x..") == 0
for _ in range(500):
    ws = {"".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 8))}
    pat = "".join(random.choice("ab.") for _ in range(random.randint(1, 3)))
    e = Answer3CountDictionary()
    for w in ws:
        e.addWord(w)
    assert e.count(pat) == sum(matches(w, pat) for w in ws)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
class Answer4MagicDictionary:
    def __init__(self):
        self.root = {}

    def buildDict(self, dictionary: List[str]) -> None:
        for w in dictionary:
            node = self.root
            for ch in w:
                node = node.setdefault(ch, {})
            node["$"] = True

    def search(self, searchWord: str) -> bool:
        def dfs(node, i, changed):
            if i == len(searchWord):
                return changed and "$" in node
            c = searchWord[i]
            if c in node and dfs(node[c], i + 1, changed):
                return True
            if not changed:
                for k, ch in node.items():
                    if k != "$" and k != c and dfs(ch, i + 1, True):
                        return True
            return False
        return dfs(self.root, 0, False)

m4 = Answer4MagicDictionary(); m4.buildDict(["hello", "leetcode"]); assert [m4.search(w) for w in ["hello", "hhllo", "hell", "leetcoded"]] == [False, True, False, False]
m4b = Answer4MagicDictionary(); m4b.buildDict(["hello", "hallo"]); assert m4b.search("hello") is True
for _ in range(500):
    ws = list({"".join(random.choice("abc") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 6))})
    q = "".join(random.choice("abc") for _ in range(random.randint(1, 3)))
    m = Answer4MagicDictionary(); m.buildDict(ws)
    exp = any(len(w) == len(q) and sum(a != b for a, b in zip(w, q)) == 1 for w in ws)
    assert m.search(q) == exp
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_find_words(board: List[List[str]], words: List[str]) -> List[str]:
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    R, C = len(board), len(board[0])
    found = []
    def dfs(r, c, node):
        ch = board[r][c]
        if ch not in node:
            return
        nxt = node[ch]
        if "$" in nxt:
            found.append(nxt.pop("$"))             # lấy ra để không thêm trùng
        board[r][c] = "#"
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and board[nr][nc] != "#":
                dfs(nr, nc, nxt)
        board[r][c] = ch
    for r in range(R):
        for c in range(C):
            dfs(r, c, root)
    return found

def can_form(board, w):
    R, C = len(board), len(board[0])
    def go(r, c, i, used):
        if board[r][c] != w[i]:
            return False
        if i == len(w) - 1:
            return True
        used.add((r, c))
        ok = any(go(r + dr, c + dc, i + 1, used) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1))
                 if 0 <= r + dr < R and 0 <= c + dc < C and (r + dr, c + dc) not in used)
        used.discard((r, c))
        return ok
    return any(go(r, c, 0, set()) for r in range(R) for c in range(C))

assert sorted(answer_5_find_words([["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]], ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
assert answer_5_find_words([["a", "b"], ["c", "d"]], ["abcb"]) == []
for _ in range(300):
    R, C = random.randint(1, 3), random.randint(1, 3)
    board = [[random.choice("ab") for _ in range(C)] for _ in range(R)]
    ws = list({"".join(random.choice("ab") for _ in range(random.randint(1, 4))) for _ in range(random.randint(1, 6))})
    exp = sorted(w for w in ws if can_form(board, w))
    assert sorted(answer_5_find_words([row[:] for row in board], ws)) == exp
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Lưu từ trong trie. `search` là DFS trên trie: chữ cái → đi đúng một con; `'.'` → thử **mọi** con (quay lui); hết mẫu → cần cờ kết thúc. Không có dấu chấm thì `O(L)`; mỗi dấu chấm nhân tối đa 26 nhánh, với ≤ 2 dấu chấm vẫn rất nhanh.

---

## Checklist trước khi nộp

- [ ] Dấu chấm thử **mọi** nhánh, không chỉ một?
- [ ] Hết mẫu thì kiểm tra cờ kết thúc?
- [ ] Không duyệt khóa cờ như một nhánh con?
- [ ] Độ dài mẫu phải khớp đúng độ dài từ?